# Fixed-tau and scaled-init controls

Ladder figures F4_10a/b/c, the recurrent-gain figure T4_10, the scaled-init dissociation F4_12, and tab_bowl.tex. Everything comes from the cached per-network records.

In [ ]:
import sys
sys.path.append("../code")
import json
import numpy as np
import matplotlib.pyplot as plt
import common as C, stats as st, plots as P
P.apply()

## data

In [ ]:
het = C.load_records("het")
hom = C.load_records("hom")
scaled = C.load_records("scaled", C.SEEDS_ORIG)
fixed = {t: C.load_records(C.RUNG_TAG[t], C.SEEDS_ORIG) for t in C.RUNGS}
fseeds = C.SEEDS_ORIG

FROZEN = "#9b8557"
RUNG_SHADE = ["#c9b98a", "#b3a071", FROZEN, "#7d6c43"]
# rungs labelled by nominal frozen value; the 2 ms rung trains at 3.26 ms (decay clamp)
RUNG_LABEL = {2.0: "2 ms", 3.5: "3.5 ms", 15.0: "15 ms", 35.0: "35 ms"}

arms = {"het": [het[s] for s in fseeds], "hom": [hom[s] for s in fseeds]}
for t in C.RUNGS:
    arms[f"fixed{t:g}"] = [fixed[t][s] for s in fseeds]
ORDER = ["het", "hom"] + [f"fixed{t:g}" for t in C.RUNGS]
COLOURS = [P.HET, P.HOM] + [FROZEN] * len(C.RUNGS)

med_tau = lambda k: float(np.median([r["tau_median_ms"] for r in arms[k]]))
LABELS = [f"Heterogeneous\n({med_tau('het'):.0f} ms)",
          f"Homogeneous\n({med_tau('hom'):.0f} ms)"] + [RUNG_LABEL[t] for t in C.RUNGS]


def vals(key, field):
    return np.array([r[field] for r in arms[key]], dtype=float)


def bar_panel(ax, field, ylabel, title, label_fs=8.5, rotate=0):
    means, errs = [], []
    for k in ORDER:
        v = vals(k, field)
        means.append(v.mean())
        errs.append(v.std(ddof=1) / np.sqrt(len(v)) * 1.96)
    x = np.arange(len(ORDER))
    ax.bar(x, means, yerr=errs, capsize=3, color=COLOURS, zorder=3)
    for k, xi in zip(ORDER, x):
        v = vals(k, field)
        ax.scatter(np.full(len(v), xi) + np.random.uniform(-0.12, 0.12, len(v)), v,
                   s=9, color="k", alpha=0.35, zorder=4, linewidths=0)
    ax.set_xticks(x)
    ax.set_xticklabels(LABELS, fontsize=label_fs, rotation=rotate, ha="right" if rotate else "center")
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left", fontsize=10.5)
    return means

## ladder: accuracy and cumulative variance

In [ ]:
np.random.seed(0)
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6))

ax = axes[0]
means = bar_panel(ax, "acc", "task accuracy", "(a) Task accuracy", label_fs=7.5, rotate=25)
P.chance_line(ax)
ax.set_ylim(0.20, max(means) * 1.18)
ax.annotate("", xy=(0, means[0]), xytext=(4, means[4]),
            arrowprops=dict(arrowstyle="<->", color=P.ACCENT, lw=1.1, shrinkA=2, shrinkB=2))
ax.text(1.9, (means[0] + means[4]) / 2, f"  {means[0] - max(means[2:]):.3f} to the best rung",
        fontsize=8, color=P.ACCENT, va="center")

ax = axes[1]
n90s = {}
for k, lbl in zip(ORDER, LABELS):
    cum = np.cumsum(np.array([r["evr_1ms"] for r in arms[k]], dtype=float), axis=1)
    m = cum.mean(axis=0)
    comps = np.arange(1, len(m) + 1)
    n90s[k] = int(np.argmax(m >= 0.90) + 1)
    flat = lbl.replace("\n", " ")
    if k in ("het", "hom"):
        h = cum.std(axis=0, ddof=1) / np.sqrt(len(cum)) * 1.96
        ax.plot(comps, m, color=P.COL[k], lw=2.0, zorder=4, label=f"{flat}, $n_{{90}}$ {n90s[k]}")
        ax.fill_between(comps, m - h, m + h, color=P.COL[k], alpha=0.18, lw=0)
    else:
        ax.plot(comps, m, color=RUNG_SHADE[ORDER.index(k) - 2], lw=1.5, ls="--",
                label=f"frozen {flat}, $n_{{90}}$ {n90s[k]}")
ax.axhline(0.90, color=P.CHANCE, ls=":", lw=1.2)
ax.set_xlim(1, 64)
ax.set_xlabel("principal component")
ax.set_ylabel("cumulative explained variance")
ax.set_title("(b) Cumulative variance", loc="left", fontsize=10.5)
ax.legend(fontsize=7, loc="lower right", framealpha=0.9)

fig.suptitle("Fixed-tau ladder: accuracy and cumulative variance",
             fontweight="bold", y=1.02, fontsize=11)
P.save(fig, C.FIGDIR / "F4_10a_fixed_tau_accuracy.png")
print("n90 by condition:", n90s)

## ladder: dimensionality

In [ ]:
np.random.seed(0)
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.8))
bar_panel(axes[0], "n90_1ms", "components for 90% variance", "(a) PCA $n_{90}$", label_fs=7.5, rotate=25)
bar_panel(axes[1], "pr_1ms", "participation ratio", "(b) Participation ratio", label_fs=7.5, rotate=25)
fig.suptitle("Fixed-tau ladder: effective dimension",
             fontweight="bold", y=1.01, fontsize=11)
P.save(fig, C.FIGDIR / "F4_10b_fixed_tau_pca.png")

## ladder: pc reconstruction

In [ ]:
np.random.seed(0)
fig, ax = plt.subplots(figsize=(7.8, 4.6))
grid = np.array(arms["het"][0]["M_GRID"], dtype=float)
for k, lbl, col in zip(ORDER, LABELS, COLOURS):
    curves = np.array([r["pc_curve"] for r in arms[k]], dtype=float)
    m = curves.mean(0)
    se = curves.std(0, ddof=1) / np.sqrt(curves.shape[0]) * 1.96
    ls = "-" if k in ("het", "hom") else "--"
    lw = 1.9 if k in ("het", "hom") else 1.2
    ax.plot(grid, m, ls, color=col, lw=lw, label=lbl.replace("\n", " "),
            zorder=3 if k in ("het", "hom") else 2)
    if k in ("het", "hom"):
        ax.fill_between(grid, m - se, m + se, color=col, alpha=0.18, lw=0)
P.chance_line(ax)
ax.set_xscale("log", base=2)
ax.set_xlabel("retained principal components $m$")
ax.set_ylabel("task accuracy through the trained readout")
ax.set_title("PC reconstruction by rank", loc="left", fontsize=10.5)
ax.legend(fontsize=8, ncol=2)
P.save(fig, C.FIGDIR / "F4_10c_fixed_tau_reconstruction.png")

for k, lbl in zip(ORDER, LABELS):
    print(f"{lbl.replace(chr(10), ' '):24s} acc {vals(k, 'acc').mean():.3f}  "
          f"n90 {vals(k, 'n90_1ms').mean():5.1f}  PR {vals(k, 'pr_1ms').mean():5.2f}  n={len(arms[k])}")

## recurrent gain

In [ ]:
def paired_slopes(ax, left, right, labels=("heterogeneous", "homogeneous")):
    for a, b in zip(left, right):
        ax.plot([0, 1], [a, b], color="#b0b0b0", lw=0.9, alpha=0.8, zorder=1)
    ax.scatter(np.zeros(len(left)), left, color=P.HET, s=32, zorder=3, label=labels[0])
    ax.scatter(np.ones(len(right)), right, color=P.HOM, s=32, zorder=3, label=labels[1])
    ax.set_xlim(-0.35, 1.35)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(labels)


acc_h, acc_o = C.vals(het, "acc"), C.vals(hom, "acc")
g_h, g_o = C.vals(het, "gain"), C.vals(hom, "gain")
ss = sorted(scaled)
g_s = np.array([scaled[s]["gain"] for s in ss])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
paired_slopes(axes[0], g_h, g_o)
axes[0].axhline(1.0, color=P.CHANCE, ls=":", lw=1.2, label="unit gain")
axes[0].set_ylabel("recurrent spectral radius")
axes[0].set_title("Recurrent gain, het vs hom")
axes[0].legend(loc="upper right")
xs = np.concatenate([g_h, g_o, g_s])
ys = np.concatenate([acc_h, acc_o, [scaled[s]["acc"] for s in ss]])
cols = [P.HET] * len(g_h) + [P.HOM] * len(g_o) + [P.SCALED] * len(ss)
axes[1].scatter(xs, ys, c=cols, s=30, zorder=3)
r_gain_acc = np.corrcoef(xs, ys)[0, 1]
P.chance_line(axes[1])
axes[1].set_xlabel("recurrent spectral radius")
axes[1].set_ylabel("task accuracy")
axes[1].set_title(f"Recurrent gain vs accuracy (r = {r_gain_acc:.2f})")
P.save(fig, C.FIGDIR / "T4_10_gain.png")

for lbl, arr in [("het", g_h), ("hom", g_o), ("scaled hom", g_s)]:
    print(f"{lbl:>10s}  mean {arr.mean():.3f}  sd {arr.std(ddof=1):.3f}  "
          f"range {arr.min():.2f} to {arr.max():.2f}  above unit gain {int((arr > 1).sum())}/{len(arr)}")

## scaled-init dissociation

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
PANELS = [("mean_rate", "mean firing rate"), ("active", "active hidden units"),
          ("acc", "task accuracy"), ("pr_1ms", "participation ratio")]
for ax, (k, lab) in zip(axes, PANELS):
    for i, (d, c) in enumerate([(het, P.HET), (hom, P.HOM), (scaled, P.SCALED)]):
        v = np.array([d[s][k] for s in ss], float)
        ax.bar(i, v.mean(), yerr=v.std(ddof=1), color=c, width=0.62, zorder=2)
        ax.scatter(np.full(len(v), i) + np.random.default_rng(0).uniform(-.13, .13, len(v)), v,
                   color="k", s=9, alpha=0.55, zorder=3)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(["het", "hom", "scaled\nhom"], fontsize=9)
    ax.set_title(lab)
    if k == "acc":
        P.chance_line(ax)
fig.suptitle("Scaled-init control: het, hom, scaled-hom", fontweight="bold", y=1.02)
P.save(fig, C.FIGDIR / "F4_12_scaled_init_dissociation.png")

## tab_bowl.tex

In [ ]:
rows = []
for t in C.RUNGS:
    key = f"fixed{t:g}"
    a = vals(key, "acc")
    b = np.array([hom[s]["acc"] for s in fseeds])
    r = st.paired_triple(a, b, f"fixed {t:g} ms vs learned hom")
    rows.append(f"{t:g} & {a.mean():.3f} $\\pm$ {a.std(ddof=1):.3f} & {vals(key, 'active').mean():.1f} & "
                f"{vals(key, 'n90_1ms').mean():.1f} & {vals(key, 'pr_1ms').mean():.2f} & {r['delta']:+.3f} & "
                f"{P.fmt_p(r['t_p'])} & {P.fmt_p(r['w_p'], r['w_floor'])} & "
                f"{P.fmt_p(r['perm_p'], r['perm_floor'])} & {r['wins']}/{r['n']} \\\\")

tex = r"""\begin{table}[H]
  \centering
  \small
  \setlength{\tabcolsep}{4pt}
  \caption{Fixed-tau ladder, ten seeds per rung: accuracy, active units, $n_{90}$ and participation
  ratio (ten-seed means), with the paired difference against the learned homogeneous baseline and the
  three paired tests at $n=10$. Rungs are named by their nominal frozen value.}
  \label{tab:bowl}
  \resizebox{\linewidth}{!}{%
  \begin{tabular}{lrrrrrrrrr}
    \toprule
    Fixed $\taumem$ (ms) & Accuracy & Active & $n_{90}$ & PR & $\Delta$ vs hom & $t$ $p$ & Wilcoxon $p$ & sign-flip $p$ & wins \\
    \midrule
    """ + "\n    ".join(rows) + r"""
    \bottomrule
  \end{tabular}}
\end{table}
"""
HDR = "% auto-generated by notebooks/04_fixed_tau_controls.ipynb; regenerate rather than edit.\n\n"
(C.TABDIR / "tab_bowl.tex").write_text(HDR + tex, encoding="utf-8")
print(tex)